# 06 — Memory lifecycle: update, delete, re-index

The earlier notebooks focused on *adding* memories. This one walks through
the rest of the lifecycle:

1. **Update** — edit an existing file and let `index_file` figure out which
   chunks actually changed (hash-based diff).
2. **Delete one chunk** — surgically remove a single chunk by UUID.
3. **Delete a file** — remove a file from disk and clean up the orphaned
   chunks that `index_path` alone will not catch.
4. **Force re-index** — rebuild every embedding for a file without changing
   it on disk (useful after swapping embedding models).

Like the other notebooks in this folder, everything runs against a
`tempfile.TemporaryDirectory()` so your real `~/.memtomem/` stays
untouched.

## Prerequisites

- **memtomem** installed:
  ```bash
  # From PyPI
  uv pip install "memtomem[ollama]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[all]" jupyter ipykernel
  ```
- **Ollama** running with `nomic-embed-text`:
  ```bash
  ollama serve
  ollama pull nomic-embed-text
  ```

In [1]:
# Verify Ollama is reachable before doing anything else.
import httpx

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
except Exception as e:
    raise RuntimeError(
        "Ollama is not reachable at http://localhost:11434.\n"
        "Start it and pull the default embedding model:\n"
        "  ollama serve\n"
        "  ollama pull nomic-embed-text\n"
        "then re-run this cell."
    ) from e

print("Ollama is up.")

Ollama is up.


## Step 1 — Set up components in a temp directory

Same boilerplate as the other notebooks: isolated sqlite DB + notes dir,
environment variable overrides, and a monkey-patch of
`load_config_overrides` so the user's real config file cannot leak in.

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"components ready. db={db_path}")

components ready. db=/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmprstjkh3f/memory.db


## Step 2 — Seed a small file and inspect its chunks

We start with a two-section markdown note, index it, and then use
`storage.list_chunks_by_source()` to see exactly which chunks landed in
the database. This is the "Read" side of lifecycle management — useful
any time you want to know what memtomem actually has on file.

In [3]:
note = notes_dir / "retrieval.md"
note.write_text(
    "# Retrieval tuning\n\n"
    "BM25 is strong on short factual queries where term overlap matters.\n\n"
    "# Rerankers\n\n"
    "A cross-encoder reranker adds latency but recovers precision on\n"
    "noisy candidate sets.\n",
    encoding="utf-8",
)

stats = await comp.index_engine.index_file(note)
print(
    f"after initial index: total={stats.total_chunks} "
    f"indexed={stats.indexed_chunks} skipped={stats.skipped_chunks} "
    f"deleted={stats.deleted_chunks}"
)

chunks = await comp.storage.list_chunks_by_source(note)
print(f"\n{len(chunks)} chunks currently stored for {note.name}:")
for c in chunks:
    heading = " / ".join(c.metadata.heading_hierarchy) or "(no heading)"
    print(f"  {str(c.id)[:8]}  {heading}")

[04/11/26 12:25:50] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=778943;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=359801;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

after initial index: total=2 indexed=2 skipped=0 deleted=0

2 chunks currently stored for retrieval.md:
  259327ae  # Retrieval tuning
  472656c9  # Rerankers


## Step 3 — Update the file and watch the hash diff

`index_file` computes a SHA-256 hash for every chunk and compares it
against what is already in the database:

- chunks whose hash matches one already in the database → `skipped_chunks`
- chunks whose hash is new (either a brand-new section or an edited one
  with a different hash) → `indexed_chunks` — these get a new UUID and a
  fresh embedding call
- chunks whose hash is no longer present in the file (either removed
  outright or replaced by an edit) → `deleted_chunks`

So an edited section shows up in *both* `indexed_chunks` (the new hash)
and `deleted_chunks` (the old hash) — the diff is hash-based, not
UUID-based.

In this step we rewrite the "Rerankers" section and add a third one
("MMR diversification"). The "Retrieval tuning" section is untouched.
Expect `indexed=2 skipped=1 deleted=1`: one skip (Retrieval tuning),
two new hashes (edited Rerankers + new MMR diversification), and one
dropped hash (the old Rerankers text).

In [4]:
note.write_text(
    "# Retrieval tuning\n\n"
    "BM25 is strong on short factual queries where term overlap matters.\n\n"
    "# Rerankers\n\n"
    "A cross-encoder reranker adds latency but recovers precision, "
    "especially when the first-stage candidate pool is noisy.\n\n"
    "# MMR diversification\n\n"
    "MMR trades a little relevance for coverage so the top-k does not "
    "collapse onto near-duplicates.\n",
    encoding="utf-8",
)

stats = await comp.index_engine.index_file(note)
print(
    f"after edit: total={stats.total_chunks} "
    f"indexed={stats.indexed_chunks} skipped={stats.skipped_chunks} "
    f"deleted={stats.deleted_chunks}"
)

chunks = await comp.storage.list_chunks_by_source(note)
print(f"\n{len(chunks)} chunks now stored:")
for c in chunks:
    heading = " / ".join(c.metadata.heading_hierarchy) or "(no heading)"
    print(f"  {str(c.id)[:8]}  {heading}")

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=900716;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=935410;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

after edit: total=3 indexed=2 skipped=1 deleted=1

3 chunks now stored:
  259327ae  # Retrieval tuning
  6d6a86b3  # Rerankers
  bfd2bf00  # MMR diversification


## Step 4 — Delete a single chunk by UUID

Sometimes you want to drop one chunk without touching the underlying
file — for example, you indexed a note that contained a secret and you
want to remove the offending section from the index immediately.

`storage.delete_chunks([uuid])` takes a list of `UUID` objects and
returns how many rows it actually deleted. The corresponding FTS and
vector rows are cleaned up in the same transaction.

In [5]:
# Pick the "MMR diversification" chunk and delete just that one.
target = next(c for c in chunks if "MMR diversification" in " ".join(c.metadata.heading_hierarchy))
print(f"deleting chunk {target.id} ({target.metadata.heading_hierarchy[0]})")

deleted = await comp.storage.delete_chunks([target.id])
print(f"delete_chunks returned: {deleted}")

remaining = await comp.storage.list_chunks_by_source(note)
print(f"\n{len(remaining)} chunks remain for {note.name}:")
for c in remaining:
    heading = " / ".join(c.metadata.heading_hierarchy) or "(no heading)"
    print(f"  {str(c.id)[:8]}  {heading}")

deleting chunk bfd2bf00-2ea4-414e-a82b-16e2a3175b25 (# MMR diversification)
delete_chunks returned: 1

2 chunks remain for retrieval.md:
  259327ae  # Retrieval tuning
  6d6a86b3  # Rerankers


## Step 5 — Delete a file and clean up orphans

Deleting a file from disk does **not** automatically remove its chunks —
`index_path` only discovers files that currently exist, so it has no way
to know that a file used to be there.

To clean up orphans you compare two sets:

- **What the database knows about**: `storage.get_all_source_files()` —
  returns the set of source-file paths currently referenced by any chunk.
- **What is actually on disk**: a filesystem walk of `notes_dir`.

Any path in the first set but not the second is an orphan, and
`storage.delete_by_source(path)` removes every chunk for that file in
one go.

In [6]:
# First, add a second file so we have something to orphan.
doomed = notes_dir / "temporary.md"
doomed.write_text(
    "# Temporary note\n\nThis file will be deleted and then cleaned up.\n",
    encoding="utf-8",
)
await comp.index_engine.index_file(doomed)

before = await comp.storage.get_all_source_files()
print(f"source files known to storage: {len(before)}")
for p in sorted(before):
    print(f"  {Path(p).name}")

# Now delete the file from disk and confirm the DB is out of sync.
doomed.unlink()

after_fs = {p.resolve() for p in notes_dir.rglob("*.md")}
after_db = await comp.storage.get_all_source_files()
orphans = {Path(p).resolve() for p in after_db} - after_fs
print(f"\norphan source files: {len(orphans)}")
for p in sorted(orphans):
    print(f"  {Path(p).name}")

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=988747;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=596140;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

source files known to storage: 2
  retrieval.md
  temporary.md

orphan source files: 1
  temporary.md


In [7]:
# Clean up each orphan. delete_by_source returns the chunk count removed.
for path in orphans:
    removed = await comp.storage.delete_by_source(path)
    print(f"delete_by_source({Path(path).name}) removed {removed} chunks")

final = await comp.storage.get_all_source_files()
print(f"\nsource files known to storage after cleanup: {len(final)}")
for p in sorted(final):
    print(f"  {Path(p).name}")

delete_by_source(temporary.md) removed 1 chunks

source files known to storage after cleanup: 1
  retrieval.md


## Step 6 — Force a full re-index of a file

The default `index_file` path uses the hash diff described in step 3, so
chunks with unchanged text are *not* re-embedded. If you deliberately
want every embedding regenerated — for example, after swapping the
embedding model — pass `force=True`. Every chunk in the file gets
re-embedded, and `indexed_chunks` will match `total_chunks`.

In [8]:
stats = await comp.index_engine.index_file(note, force=True)
print(
    f"force re-index: total={stats.total_chunks} "
    f"indexed={stats.indexed_chunks} skipped={stats.skipped_chunks} "
    f"deleted={stats.deleted_chunks}"
)

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=532251;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=227864;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

force re-index: total=3 indexed=3 skipped=0 deleted=0


## In production: the `FileWatcher` does this for you

The explicit calls in this notebook are the *manual* version of what the
MCP server's `FileWatcher` does automatically. When the server is
running, the watcher observes every configured memory directory and:

- re-runs `index_file` when a file is modified (hash diff → incremental
  re-embedding),
- calls `delete_by_source` when a file is removed (so orphans are
  cleaned up immediately).

Notebooks prefer explicit calls because they are deterministic: you can
assert exactly what changed on each step, without racing the watcher's
debounce timer. Once you move into a long-running agent process, let the
watcher handle the routine updates and reserve `delete_chunks` /
`delete_by_source` for surgical cases.

## Cleanup

In [9]:
await close_components(comp)
tmp.cleanup()

for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
